# Loading libraries


In [ ]:
import pandas as pd
import numpy as np

# Loading datasets

### Labeled ClinPGx

In [ ]:
labeled_clinpgx_dataset = pd.read_csv('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/labeled_clinpgx_dataset.csv',
                               sep="\t")

labeled_clinpgx_dataset = labeled_clinpgx_dataset.drop(columns='User_input_1')

labeled_clinpgx_dataset.drop_duplicates(inplace=True)


### Unlabeled Populational

In [ ]:
unlabeled_populational_dataset = pd.read_csv('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/unlabeled_populational_dataset.csv',
                               sep="\t")

unlabeled_populational_dataset = unlabeled_populational_dataset.drop(columns='User_input_1')

unlabeled_populational_dataset.drop_duplicates(inplace=True)


# Inspect missings

### **Labeled ClinPGx dataset**

In [ ]:
## Delete rows that only contain NaN values

cols_to_check = [c for c in labeled_clinpgx_dataset.columns if c != 'User_input']

labeled_clinpgx_dataset_rmna = labeled_clinpgx_dataset[labeled_clinpgx_dataset[cols_to_check].notna().any(axis=1)]

In [ ]:
def missing_per_col(df):
  print(
    (df.isna().mean() * 100)
    .sort_values(ascending=False)
    .to_string()
    )

In [ ]:
missing_per_col(labeled_clinpgx_dataset_rmna)

In [ ]:
## Delete columns with high percentage of missing values
## I chose 50% of missing values as the threshold after inspection of the % of missings per column

def del_high_missing_cols(df, threshold):
  cols_to_keep = (
    df.isna().mean().mul(100)
    .loc[lambda x: x < threshold]
    .index
    .tolist()
  )
  df_cleaned = df[cols_to_keep]
  return df_cleaned


In [ ]:
labeled_clinpgx_dataset_rmna = del_high_missing_cols(labeled_clinpgx_dataset_rmna, 50)

In [ ]:
def missing_per_row(df):
  missing_pct_per_row = df.isnull().mean(axis=1) * 100
  thresholds = [10, 20, 30, 50, 75, 90]
  for t in thresholds:
      bad_rows_count = (missing_pct_per_row > t).sum()

      pct_of_dataset = (bad_rows_count / len(df)) * 100

      print(f"{t}% missing: {bad_rows_count} rows ({pct_of_dataset:.2f}% of dataset)")



In [ ]:
missing_per_row(labeled_clinpgx_dataset_rmna)

In [ ]:
def rem_high_miss_pct_rows(df, max_missing_allowed):
  missing_pct_per_row = df.isnull().mean(axis=1) * 100
  df_cleaned = df[missing_pct_per_row <= max_missing_allowed]

  print(f"Original shape: {df.shape}")
  print(f"Cleaned shape: {df_cleaned.shape}")

  return df_cleaned


In [ ]:
labeled_clinpgx_dataset_rmna = rem_high_miss_pct_rows(labeled_clinpgx_dataset_rmna, 75)

### **Unlabeled Populational dataset**

In [ ]:
## Delete rows that only contain NaN values

cols_to_check = [c for c in unlabeled_populational_dataset.columns if c != 'User_input']

unlabeled_populational_dataset_rmna = unlabeled_populational_dataset[unlabeled_populational_dataset[cols_to_check].notna().any(axis=1)]

In [ ]:
missing_per_col(unlabeled_populational_dataset_rmna)

In [ ]:
unlabeled_populational_dataset_rmna = del_high_missing_cols(unlabeled_populational_dataset_rmna, 50)

In [ ]:
missing_per_row(unlabeled_populational_dataset_rmna)

In [ ]:
unlabeled_populational_dataset_rmna = rem_high_miss_pct_rows(unlabeled_populational_dataset_rmna, 75)

# Final dataset

In [ ]:
labeled_clinpgx_dataset_rmna['label'] = 1
unlabeled_populational_dataset_rmna['label'] = -1

In [ ]:
common_cols = labeled_clinpgx_dataset_rmna.columns.intersection(
    unlabeled_populational_dataset_rmna.columns
    ).to_list()

In [ ]:
def subset_for_common(df):
  df = df[common_cols]
  return df


labeled_clinpgx_dataset_rmna = subset_for_common(labeled_clinpgx_dataset_rmna)
unlabeled_populational_dataset_rmna = subset_for_common(unlabeled_populational_dataset_rmna)

In [ ]:
full_dataset = pd.concat([labeled_clinpgx_dataset_rmna, unlabeled_populational_dataset_rmna], axis=0)

In [ ]:
full_dataset.to_csv('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/ml_dataset.csv', index=False)